# Qwen3.5-9B 导出后 Calibration + AWQ 量化流程

本 Notebook 用于：
1. 配置环境（`uv`）
2. 从 SFT 数据生成 `calibration.jsonl`
3. 对导出模型执行 AWQ 4bit 量化
4. 用 vLLM 以 OpenAI API 格式启动并测试

## 0) 路径约定

请先确认以下路径：
- 已导出的合并模型目录：`./export/qwen3.5-9b-ft-merged`
- 训练集文件（json 或 jsonl）：例如 `./data/weclone_sft/chat-sft.json`
- 根目录已有脚本：`build_calibration_jsonl.py`、`quantize_awq.py`

In [ ]:
# 你可以按需修改这些变量
DATASET_PATH = "./data/weclone_sft/chat-sft.json"
MERGED_MODEL_PATH = "./export/qwen3.5-9b-ft-merged"
CALIB_FILE = "./calibration.jsonl"
AWQ_MODEL_PATH = "./export/qwen3.5-9b-ft-merged-awq"

print('DATASET_PATH =', DATASET_PATH)
print('MERGED_MODEL_PATH =', MERGED_MODEL_PATH)
print('CALIB_FILE =', CALIB_FILE)
print('AWQ_MODEL_PATH =', AWQ_MODEL_PATH)

## 1) 配置环境（uv）

按你要求，使用 `uv pip install --python /path/to/venv/bin/python` 语法。

如果你在 Colab / Linux：
- 先创建 venv：`uv venv /path/to/venv`
- 再安装依赖：`uv pip install --python /path/to/venv/bin/python ...`

In [ ]:
!uv --version
!uv venv /content/awq-venv
!uv pip install --python /content/awq-venv/bin/python -U autoawq transformers datasets accelerate safetensors vllm

## 2) 生成 calibration.jsonl

建议先用 200 条快速验证流程，再提升到 512 或 1000。

In [ ]:
!python build_calibration_jsonl.py \
  --input {DATASET_PATH} \
  --output {CALIB_FILE} \
  --num-samples 512

!head -n 3 {CALIB_FILE}

## 3) AWQ 量化（4bit）

默认参数：
- `w_bit=4`
- `q_group_size=128`
- `max_calib_samples=512`
- `max_calib_seq_len=1024`

In [ ]:
!python quantize_awq.py \
  --model-path {MERGED_MODEL_PATH} \
  --calib-file {CALIB_FILE} \
  --output-path {AWQ_MODEL_PATH} \
  --w-bit 4 \
  --q-group-size 128 \
  --max-calib-samples 512 \
  --max-calib-seq-len 1024

## 4) 启动 vLLM（OpenAI API）

启动后接口地址：`http://127.0.0.1:8000/v1`

In [ ]:
!vllm serve {AWQ_MODEL_PATH} \
  --quantization awq \
  --dtype float16 \
  --served-model-name qwen35-9b-ft-awq \
  --host 0.0.0.0 \
  --port 8000 \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.92

## 5) OpenAI API 格式测试

请在 vLLM 服务启动后执行该单元。

In [ ]:
!curl http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen35-9b-ft-awq","messages":[{"role":"user","content":"你好，做个自我介绍"}],"temperature":0.7}'

## 6) 常见问题

- OOM：把 `--max-model-len` 从 2048 降到 1024。
- 量化慢：先把 `--max-calib-samples` 降到 200 验证流程。
- 效果下降明显：增加校准样本数量，并使用更贴近业务的数据分布。